# Demo: Semantic Kernel의 자동 펑션 호출 기능

## 패키지 설치

In [ ]:
#r "nuget: Microsoft.Extensions.Configuration.Json, 9.*"
#r "nuget: Microsoft.Extensions.AI, 9.*-*"
#r "nuget: Microsoft.Extensions.DependencyInjection.Abstractions, 9.*"
#r "nuget: Microsoft.SemanticKernel, 1.*"
#r "nuget: Microsoft.SemanticKernel.Plugins.Core, 1.*-*"


## 네임스페이스 지정

In [ ]:
using System.ClientModel;
using System.IO;

using Microsoft.Extensions.Configuration;
using Microsoft.Extensions.AI;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using Microsoft.SemanticKernel.Connectors.OpenAI;

using OpenAI;


## 액세스 키 로드

In [ ]:
var path = Path.Combine(Directory.GetCurrentDirectory(), "appsettings.Development.json");
var config = new ConfigurationBuilder().AddJsonFile(path).Build();

## Azure OpenAI 인스턴스 생성

In [ ]:
var kernel = Kernel.CreateBuilder()
                   .AddAzureOpenAIChatCompletion(
                       endpoint: config["Azure:OpenAI:Endpoint"],
                       apiKey: config["Azure:OpenAI:Token"],
                       deploymentName: config["Azure:OpenAI:ModelName"])
                   .Build();

## 플러그인 코드 로드

In [ ]:
#!import Plugins/Appointment.cs
#!import Plugins/BookingsPlugin.cs


## 플러그인 임포트

In [ ]:
kernel.Plugins.AddFromType<BookingsPlugin>();


## 챗봇 생성

In [ ]:
// Get chat completion service
var chatCompletionService = kernel.GetRequiredService<IChatCompletionService>();

// Start the conversation
string? input = null;


## 챗봇 자동 펑션 호출 기능 설정

In [ ]:
// Enable function calling behaviour
var executionSettings = new PromptExecutionSettings
{
    // FunctionChoiceBehavior = FunctionChoiceBehavior.Auto()
    FunctionChoiceBehavior = FunctionChoiceBehavior.None()
};

## 챗봇 실행

In [ ]:
// Run assistant
ChatHistory chatHistory = [];

while (true)
{
    Console.Write("User > ");

    try
    {
        input = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("무엇을 도와드릴까요?");
    }
    catch (Exception ex)
    {
        Console.WriteLine("이용해 주셔서 감사합니다.");
        break;
    }

    chatHistory.AddUserMessage(input);

    // Get the result from the AI
    var result = await chatCompletionService.GetChatMessageContentAsync(chatHistory, executionSettings, kernel);

    // Print the result
    Console.WriteLine("Assistant > " + result);

    // Add the message from the agent to the chat history
    chatHistory.AddMessage(result.Role, result?.Content!);
}
